# **Lazy Initialization**
:label:`sec_lazy_init`

- So far, we have **set up networks** with some **unintuitive practices**:
  - Defining network architectures **without specifying the input dimensionality**.
  - Adding layers **without specifying the output dimension** of the previous layer.
  - "Initializing" parameters **before knowing the model size**.

- **Why does this work?**
  - The framework **defers initialization** until the **first time data is passed through the model**.
  - Sizes of each layer are **inferred on the fly**.

- **Benefits of lazy initialization**:
  - Particularly useful in **convolutional neural networks (CNNs)**:
    - Input dimensionality (e.g., image resolution) **affects subsequent layers**.
    - Avoids the need to **specify dimensions during model definition**.
    - Simplifies **specification and modification** of models.

- Next, we explore the **mechanics of initialization** in detail.


In [1]:
import torch
from torch import nn
from d2l import torch as d2l

- To begin, let's instantiate an MLP.


In [2]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))

- At this point, the network **cannot determine** the dimensions of the **input layer's weights**:
  - The **input dimension** remains **unknown**.

- **Consequence**:
  - The framework **has not initialized any parameters** yet.

- We can **confirm this** by attempting to **access the parameters** below.


In [3]:
net[0].weight

<UninitializedParameter>

- Next let's pass data through the network
to make the framework finally initialize parameters.


In [4]:
X = torch.rand(2, 20)
net(X)

net[0].weight.shape

torch.Size([256, 20])

- As soon as the **input dimensionality** is known (**20** in this case):
  - The framework **identifies the shape** of the **first layer's weight matrix** by using the value of **20**.
  - After recognizing the first layer's shape, the framework:
    - Proceeds to the **second layer**.
    - Continues **sequentially through the computational graph** until all shapes are known.

- **Lazy initialization process**:
  - Only the **first layer** requires lazy initialization.
  - The framework **initializes sequentially**, layer by layer.
  - Once all shapes are identified, the framework **initializes the parameters**.

- **Dry run method**:
  - Involves passing **dummy inputs through the network**.
  - Helps **infer all parameter shapes**.
  - Useful when **default random initializations are not desired**.


In [5]:
@d2l.add_to_class(d2l.Module)  #@save
def apply_init(self, inputs, init=None):
    self.forward(*inputs)
    if init is not None:
        self.net.apply(init)

## **Summary**

- **Lazy initialization** allows the framework to **infer parameter shapes automatically**.
- Benefits of lazy initialization:
  - **Convenience** in modifying architectures.
  - Eliminates a **common source of errors** related to mismatched dimensions.
- To **initialize parameters**, we **pass data through the model**.
